In [14]:
import numpy as np
import pandas as pd

from sklearn import set_config
set_config(display="text")

from sklearn.naive_bayes import BernoulliNB
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.metrics import accuracy_score

# 문제 정의

##### 베르누이 나이브베이즈 분류 모델을 사용하여 스팸 메일을 분류.
##### 데이터가 범주(카테고리)값을 가지고 있는 경우, 시행 횟수와 상관 없음

# 데이터 수집

##### 이번 실습에서는 간단한 스팸 메일 분류 실습을 위해 아래 이메일 타이틀과 스팸 여부가 있는 데이터를 사용.

In [15]:
email_list = [
                {'email title': 'free game only today', 'spam': True},
                {'email title': 'cheapest flight deal', 'spam': True},
                {'email title': 'limited time offer only today only today', 'spam': True},
                {'email title': 'today meeting schedule', 'spam': False},
                {'email title': 'your flight schedule attached', 'spam': False},
                {'email title': 'your credit card statement', 'spam': False}
             ]


df = pd.DataFrame(email_list)
df

,email title,spam
0,free game only today,True
1,cheapest flight deal,True
2,limited time offer only today only today,True
3,today meeting schedule,False
4,your flight schedule attached,False
5,your credit card statement,False


# 데이터 다듬기

##### sklearn의 베르누이 나이브베이즈 분류기는 숫자만을 다루기 때문에, True와 False를 1과 0으로 치환.

In [16]:
df['label'] = df['spam'].map({True:1, False:0})
df

,email title,spam,label
0,free game only today,True,1
1,cheapest flight deal,True,1
2,limited time offer only today only today,True,1
3,today meeting schedule,False,0
4,your flight schedule attached,False,0
5,your credit card statement,False,0


In [17]:
# 학습에 사용될 데이터와 분류값을 나눔.

df_x = df['email title']
df_y = df['label'] # 정답 데이터

In [ ]:
# email title 문자열로 제공 : 알고리즘의 입력 데이터로 사용 불가
# sklearn 에서 문자열을 숫자로 변환
# CountVectorizer 

cv = CountVectorizer(binary=True) # CountVectorizer 클래스 : 0, 1로 반환 
# 1. 모든 단어를 열거한 다음 정렬 한다(fit, 사전을 만듬) 2. 문자열 -> 숫자로 변환(transform) 중첩 데이터는 제외, 입력데이터 크기를 동일하게 맞춤  
x_traincv = cv.fit_transform(df_x) 
x_traincv

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 23 stored elements and shape (6, 17)>

In [ ]:
cv.get_feature_names_out()  # 알파벳 순의 단어

# [모든 단어를 열거한 다음 정렬 한다]

array(['attached', 'card', 'cheapest', 'credit', 'deal', 'flight', 'free',
       'game', 'limited', 'meeting', 'offer', 'only', 'schedule',
       'statement', 'time', 'today', 'your'], dtype=object)

In [20]:
encoded_input = x_traincv.toarray()
encoded_input

array([[0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0],
       [0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0],
       [1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1],
       [0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1]])

In [21]:
# inverse_transform : 숫자 -> 문자열
# fit_transform : 문자열 -> 숫자 
cv.inverse_transform(encoded_input[[0]])

[array(['free', 'game', 'only', 'today'], dtype='<U9')]

## 베르누이 나이브베이즈 분류 모델 생성

In [22]:
# 학습 데이터로 베르누이 분류기를 학습.

bnb = BernoulliNB()
y_train = df_y.astype('int')  # label type을 정수형으로 선언

bnb.fit(x_traincv, y_train) # x_traincv : 학습 데이터, y_train 정답 데이터

# fit() 하면 예측 모델이 생성

BernoulliNB()

In [30]:
# 테스트 데이터 다듬기
test_email_list = [
                {'email title': 'free flight offer', 'spam': True},
                {'email title': 'hey traveler free flight deal', 'spam': True},
                {'email title': 'limited free game offer', 'spam': True},
                {'email title': 'today flight schedule', 'spam': False},
                {'email title': 'your credit card attached', 'spam': False},
                {'email title': 'free credit card offer only today', 'spam': False}
             ]

test_df = pd.DataFrame(test_email_list)
test_df['label'] = test_df['spam'].map({True:1, False:0})
test_df

,email title,spam,label
0,free flight offer,True,1
1,hey traveler free flight deal,True,1
2,limited free game offer,True,1
3,today flight schedule,False,0
4,your credit card attached,False,0
5,free credit card offer only today,False,0


In [ ]:
test_x = test_df['email title']
test_y = test_df['label']  # 정답 데이터

In [31]:
# 학습 데이터는 이미 만들어져 있음(fit_transform() 함수) -> 단어 사전을 다시 만들 필요는 없음, 테스트 데이터는 문자열을 숫자로 변환(transform()함수 호출) 

x_testcv = cv.transform(test_x) 
x_testcv

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 23 stored elements and shape (6, 17)>

## 테스트

In [32]:
predicted = bnb.predict(x_testcv)

## 정확도

In [33]:
accuracy_score(test_y, predicted)

0.8333333333333334